# 6.2c - TF-IDF to Layer 3b bridge

This notebook rebuilds the pooled 5-fold TF-IDF prediction for `score_std_3way` and exports a meeting-level file for the Layer 3b regressions.

Why this notebook exists:
- it isolates the text-only predictor used for Layer 3b
- it keeps the sample logic transparent
- every meeting's prediction is out-of-sample at the TF-IDF stage because predictions are pooled from held-out folds only

In [1]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline

In [2]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

ROOT = Path("/content") if IN_COLAB else Path("..").resolve()
OUTPUT_DIR = ROOT / "output" / "stance"
MERGED_TARGET_PATH = OUTPUT_DIR / "turn_disagreement_targets.csv"
SGKF_FOLD_PATH = OUTPUT_DIR / "disagreement_sgkf_folds.csv"
EXPORT_DOC_PATH = OUTPUT_DIR / "tfidf_pred_score_std_3way_cv5_doc.csv"
EXPORT_TURN_PATH = OUTPUT_DIR / "tfidf_pred_score_std_3way_cv5_turn.csv"

RANDOM_STATE = 42
N_SPLITS_CV = 5

tfidf_reg = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=5, stop_words="english", sublinear_tf=True)),
    ("ridge", Ridge(alpha=3.0, solver="lsqr")),
])

print(f"ROOT: {ROOT}")
print(f"MERGED_TARGET_PATH exists: {MERGED_TARGET_PATH.exists()}")
print(f"SGKF_FOLD_PATH exists: {SGKF_FOLD_PATH.exists()}")

ROOT: C:\Users\sffra\Projects\BSE 2025-2026\central-bank-spillovers
MERGED_TARGET_PATH exists: True
SGKF_FOLD_PATH exists: True


## 1. Load disagreement targets

In [3]:
disagreement = pd.read_csv(MERGED_TARGET_PATH)
if "doc_id" not in disagreement.columns:
    disagreement["doc_id"] = disagreement["turn_uid"].str.rsplit("_", n=1).str[0]

print(f"Turns: {len(disagreement):,}")
print(f"Meetings: {disagreement['doc_id'].nunique()}")
display(disagreement[["turn_uid", "doc_id", "bank", "text", "score_std_3way"]].head())

Turns: 3,972
Meetings: 206


,turn_uid,doc_id,bank,text,score_std_3way
0,BoE_201502_transcript_1,BoE_201502_transcript,BoE,"Well, recognising that this is a hypothetical ...",0.000000
1,BoE_201502_transcript_3,BoE_201502_transcript,BoE,"Well, as you know, Jennifer, we don't do real ...",0.000000
2,BoE_201502_transcript_5,BoE_201502_transcript,BoE,Look - markets will - I think - let me go back...,0.000000
3,BoE_201502_transcript_7,BoE_201502_transcript,BoE,"Well, I think it's pretty clear, in terms of o...",0.000000
4,BoE_201502_transcript_9,BoE_201502_transcript,BoE,"Well, speaking from an inflation perspective, ...",0.372678


## 2. Ensure locked 5-fold meeting assignments

This matches the `6.2` logic: group at the meeting level, stratify by `bank × disagreement bin`, and lock assignments to CSV.

In [4]:
meeting_df = (
    disagreement.groupby("doc_id")
    .agg(
        bank=("bank", "first"),
        date=("date", "first"),
        n_turns=("turn_uid", "count"),
        mean_p_directional=("p_directional", "mean"),
        share_split=("split", "mean"),
    )
    .reset_index()
)
meeting_df["disagree_bin"] = pd.qcut(
    meeting_df["mean_p_directional"], q=3, labels=["low", "med", "high"], duplicates="drop"
)
meeting_df["strata"] = meeting_df["bank"] + "_" + meeting_df["disagree_bin"].astype(str)

if SGKF_FOLD_PATH.exists():
    fold_map = pd.read_csv(SGKF_FOLD_PATH)
    print(f"Loaded fold assignments from {SGKF_FOLD_PATH}")
else:
    min_stratum = meeting_df["strata"].value_counts().min()
    assert min_stratum >= N_SPLITS_CV, f"Stratum too small for {N_SPLITS_CV}-fold CV: {min_stratum}"
    skf = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)
    meeting_df["cv_fold"] = -1
    for fold, (_, test_idx) in enumerate(skf.split(meeting_df, y=meeting_df["strata"])):
        meeting_df.iloc[test_idx, meeting_df.columns.get_loc("cv_fold")] = fold
    fold_map = meeting_df[["doc_id", "cv_fold"]].copy()
    fold_map.to_csv(SGKF_FOLD_PATH, index=False)
    print(f"Created and saved fold assignments -> {SGKF_FOLD_PATH}")

disagreement = disagreement.merge(fold_map, on="doc_id", how="left")
assert disagreement["cv_fold"].notna().all(), "Missing cv_fold assignments"
assert disagreement.groupby("doc_id")["cv_fold"].nunique().eq(1).all(), "Meeting appears in multiple folds"

print(disagreement[["doc_id", "cv_fold"]].drop_duplicates()["cv_fold"].value_counts().sort_index().rename("n_meetings"))

Loaded fold assignments from C:\Users\sffra\Projects\BSE 2025-2026\central-bank-spillovers\output\stance\disagreement_sgkf_folds.csv
cv_fold
0    42
1    41
2    41
3    41
4    41
Name: n_meetings, dtype: int64


## 3. Pooled 5-fold TF-IDF prediction for `score_std_3way`

Each fold fits TF-IDF + Ridge on 4/5 of the meetings and predicts only the held-out 1/5. The pooled prediction is therefore out-of-sample for every meeting.

In [5]:
pred_series = pd.Series(np.nan, index=disagreement.index, dtype=float)
fold_rows = []

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

for fold in range(N_SPLITS_CV):
    train_df = disagreement[disagreement["cv_fold"] != fold].copy()
    test_df = disagreement[disagreement["cv_fold"] == fold].copy()
    assert set(train_df["doc_id"]).isdisjoint(set(test_df["doc_id"])), f"Fold {fold}: meeting leakage"

    model = tfidf_reg.fit(train_df["text"], train_df["score_std_3way"].values)
    pred = model.predict(test_df["text"])
    pred_series.loc[test_df.index] = pred

    rho = spearmanr(test_df["score_std_3way"], pred).statistic
    fold_rows.append({
        "fold": fold,
        "n_meetings": int(test_df["doc_id"].nunique()),
        "n_turns": int(len(test_df)),
        "spearman": float(rho) if not math.isnan(rho) else np.nan,
        "mae": mean_absolute_error(test_df["score_std_3way"], pred),
        "rmse": rmse(test_df["score_std_3way"], pred),
        "r2": r2_score(test_df["score_std_3way"], pred),
    })
    print(f"Fold {fold} done - test meetings: {test_df['doc_id'].nunique()}")

disagreement["pred_score_std_3way_tfidf_cv5"] = pred_series.values
assert disagreement["pred_score_std_3way_tfidf_cv5"].notna().all(), "Missing pooled predictions"

fold_df = pd.DataFrame(fold_rows)
fold_df

Fold 0 done - test meetings: 42
Fold 1 done - test meetings: 41
Fold 2 done - test meetings: 41
Fold 3 done - test meetings: 41
Fold 4 done - test meetings: 41


,fold,n_meetings,n_turns,spearman,mae,rmse,r2
0,0,42,826,0.371370,0.167915,0.202691,0.128830
1,1,41,785,0.399432,0.171650,0.211661,0.145747
2,2,41,764,0.460680,0.166148,0.200206,0.191542
3,3,41,796,0.397922,0.170524,0.206068,0.144742
4,4,41,801,0.409024,0.168629,0.203635,0.159523


## 4. Meeting-level handoff frame

This is the Layer 3b bridge: aggregate turn-level actual and predicted disagreement to the meeting level, then merge on `doc_id` into the shock data.

In [6]:
doc_pred = (
    disagreement.groupby(["bank", "doc_id"], as_index=False)
    .agg(
        date=("date", "first"),
        actual_score_std_3way=("score_std_3way", "mean"),
        pred_std3=("pred_score_std_3way_tfidf_cv5", "mean"),
        n_turns=("turn_uid", "size"),
        cv_fold=("cv_fold", "first"),
    )
    .sort_values(["date", "bank", "doc_id"])
    .reset_index(drop=True)
)

turn_rho = spearmanr(disagreement["score_std_3way"], disagreement["pred_score_std_3way_tfidf_cv5"]).statistic
meeting_rho = spearmanr(doc_pred["actual_score_std_3way"], doc_pred["pred_std3"]).statistic

print(f"Turn-level Spearman:    {turn_rho:.3f}")
print(f"Meeting-level Spearman: {meeting_rho:.3f}  (n={len(doc_pred)} meetings)")
display(doc_pred.head())

Turn-level Spearman:    0.407
Meeting-level Spearman: 0.544  (n=206 meetings)


,bank,doc_id,date,actual_score_std_3way,pred_std3,n_turns,cv_fold
0,BoE,BoE_201502_transcript,201502,0.117341,0.142448,25,4
1,BoE,BoE_201505_transcript,201505,0.165424,0.141676,24,3
2,BoE,BoE_201508_transcript,201508,0.154415,0.144417,23,0
3,BoE,BoE_201511_transcript,201511,0.150456,0.156793,21,0
4,BoE,BoE_201602_transcript,201602,0.108029,0.149139,20,2


In [7]:
disagreement.to_csv(EXPORT_TURN_PATH, index=False)
doc_pred.to_csv(EXPORT_DOC_PATH, index=False)
print(f"Saved turn-level pooled predictions -> {EXPORT_TURN_PATH}")
print(f"Saved doc-level Layer 3b handoff    -> {EXPORT_DOC_PATH}")

Saved turn-level pooled predictions -> C:\Users\sffra\Projects\BSE 2025-2026\central-bank-spillovers\output\stance\tfidf_pred_score_std_3way_cv5_turn.csv
Saved doc-level Layer 3b handoff    -> C:\Users\sffra\Projects\BSE 2025-2026\central-bank-spillovers\output\stance\tfidf_pred_score_std_3way_cv5_doc.csv


## 5. How this feeds Layer 3b

Use `doc_pred` or the exported CSV as the text-predicted disagreement regressor:

```python
import statsmodels.api as sm

pred_std3_df = pd.read_csv("../output/stance/tfidf_pred_score_std_3way_cv5_doc.csv")
df = shock_df.merge(pred_std3_df[["doc_id", "pred_std3"]], on="doc_id", how="inner")
X = sm.add_constant(df["pred_std3"])

for dep in ["shock_sd", "shock_range"]:
    res = sm.OLS(df[dep], X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
    print(f"\n{dep}")
    print(res.summary().tables[1])
```

Interpretation:
- `pred_std3` is text-only at inference time
- every meeting's TF-IDF prediction is out-of-sample at the prediction stage
- this avoids the sample issue from the old full-sample CLS/Ridge construction